List of the functions implemented in this notebook for the implementation of the "No-U-turn" algorithm for Hamiltonian Monte Carlo sampling:
1. L example (gasussian log-density) -> OK
2. Leapfrog -> Ok
3. pseudo_energy -> OK
4. FindReasonableEpsilon -> OK
5. BuildTree + auxiliary functions -> OK
6. NUTS for generic L -> up to TEST

The functions are implemented in JAX in order to exploit its capabilitied to speed up the computations.

In [4]:
import math 
import numpy as np
import jax.random as jrnd
import jax.numpy as jnp
from jax import grad, jit, jax, lax
from numpy.typing import NDArray
from typing import Callable, NamedTuple, Tuple

# from functools import partial

In [5]:
# Updated Leapfrog integrator, using user defined function 
@jax.jit( static_argnums = (0,) )
def Leapfrog(
    L      : Callable,
    theta  : jnp.ndarray,
    r      : jnp.ndarray,
    epsilon: float) -> Tuple[jnp.ndarray, jnp.ndarray]:

    # Shape discipline
    assert theta.ndim == 1
    assert r.ndim == 1

    # Make epsilon a JAX array to avoid recompilation
    epsilon = jnp.asarray(epsilon)
    
    # half-step momentum update
    grad0 = grad(L)(theta,)
    r_tilde = r + 0.5 * epsilon * grad0

    # full-step position update
    theta_tilde = theta + epsilon * r_tilde

    # second half-step momentum update
    grad1 = grad(L)(theta_tilde)
    r_tilde = r_tilde + 0.5 * epsilon * grad1

    return theta_tilde, r_tilde


# This calculate a value proportional to log(p(r,theta)) that is the hamiltonian value at r,theta;
# Working in log-space provide more stability and is useful in order to avoid overflows

@jax.jit( static_argnums = (0,) )
def log_p(
    L     : Callable,
    theta : jnp.ndarray, 
    r     : jnp.ndarray) -> float:

    # Shape discipline
    assert theta.ndim == 1
    assert r.ndim == 1

    # "potential" + "kinetic"
    E = L(theta) - ( 0.5 * jnp.vdot(r, r) )
    return E

# Alghorithm 4 of the article

@jax.jit( static_argnums = (0,) )
def FindReasonableEpsilon(
    L     : Callable,
    theta : jnp.ndarray, 
    key   =jnp.array([0, 0])) -> float:
    
    assert theta.ndim == 1

    # sample momentum r ~ N(0, I)
    key, subkey = jrnd.split(key)
    r = jrnd.normal(subkey, shape=theta.shape)

    # initial epsilon
    epsilon = 1.0

    # compute initial log joint density
    E0 = log_p(L, theta, r)

    # one leapfrog step
    theta_prime, r_prime = Leapfrog(L, theta, r, epsilon)
    E1 = log_p(L, theta_prime, r_prime)

    # log acceptance probability
    log_accept = E1 - E0

    # direction: +1 (increase) or -1 (decrease)
    a = jnp.where(log_accept > jnp.log(0.5), 1.0, -1.0)

    # -------------------------------
    # cond_fun: while loop consition
    # -------------------------------
    def cond_fun(state):
        eps, log_acc = state
        return (a * log_acc) > (-a * jnp.log(2.0))

    # --------------------
    # body_fun: loop body
    # --------------------
    def body_fun(state):
        eps, _ = state
        eps = eps * (2.0 ** a)
        theta_p, r_p = Leapfrog( L, theta, r, eps)
        E1 = log_p(L, theta_p, r_p)
        log_acc = E1 - E0
        return (eps, log_acc)

    # while loop in JAX
    epsilon, _ = jax.lax.while_loop(cond_fun, body_fun, (epsilon, log_accept))

    return epsilon

# input: [theta, r, u, v, j, epsilon, theta_0, r_0]
# theta,r -> coordinates
# u -> slice variable
# v -> {+1,-1} direction
# j -> number of previous doublings

class Root(NamedTuple):
    theta   : jnp.ndarray
    r       : jnp.ndarray
    u       : float
    v       : int
    j       : int
    epsilon : float
    theta_0 : jnp.ndarray
    r_0     : jnp.ndarray

# Define a class to mange the tree variables
class Tree(NamedTuple):
    theta_minus : jnp.ndarray
    r_minus     : jnp.ndarray
    theta_plus  : jnp.ndarray
    r_plus      : jnp.ndarray
    theta_prime : jnp.ndarray
    n_prime     : jnp.ndarray
    s_prime     : jnp.ndarray
    alpha_prime : jnp.ndarray
    n_a_prime   : jnp.ndarray


@jax.jit( static_argnums = (0,) )
def build_tree_base(
    L     : Callable,
    root  : Root) -> Tree:
    
    theta, r = root.theta, root.r
    u, v = root.u, root.v
    eps = root.epsilon
    theta0, r0 = root.theta_0, root.r_0

    theta_p, r_p = Leapfrog( L, theta, r, v * eps)

    logu = jnp.log(u)
    lp = log_p(L, theta_p, r_p)

    n_p = (logu <= lp).astype(jnp.int32)
    s_p = (logu < (1000.0 + lp)).astype(jnp.int32)

    delta = (
        L(theta_p) - 0.5 * jnp.dot(r_p, r_p)
        - L(theta0) + 0.5 * jnp.dot(r0, r0)
    )
    alpha_p = jnp.minimum(1.0, jnp.exp(delta))
    n_a_p = jnp.array(1, dtype=jnp.int32)

    return Tree(
        theta_minus=theta_p,
        r_minus=r_p,
        theta_plus=theta_p,
        r_plus=r_p,
        theta_prime=theta_p,
        n_prime=n_p,
        s_prime=s_p,
        alpha_prime=alpha_p,
        n_a_prime=n_a_p,
    )

# function to verify if the stop condition is reached
@jax.jit
def stop_criterion(theta_minus, theta_plus, r_minus, r_plus) -> jnp.int32:
    delta = theta_plus - theta_minus
    cond = (jnp.dot(delta, r_minus) >= 0) & (jnp.dot(delta, r_plus) >= 0)
    return cond.astype(jnp.int32)

def BuildTree(
    L    : Callable, 
    root : Root, 
    key ):
    """
    Pure-Python recursive BuildTree (no JAX control flow).
    """

    # Base case: j == 0
    if root.j == 0:
        return build_tree_base( L, root), key

    # Recursive case: j > 0

    # --- 1. Left subtree ---
    left_root = root._replace(j=root.j - 1)
    left_tree, key_left = BuildTree( L, left_root, key)

    # --- 2. Right subtree only if still valid ---
    if left_tree.s_prime == 1:
        if root.v == -1:
            new_root = root._replace(
                theta=left_tree.theta_minus,
                r=left_tree.r_minus,
                j=root.j - 1,
            )
        else:
            new_root = root._replace(
                theta=left_tree.theta_plus,
                r=left_tree.r_plus,
                j=root.j - 1,
            )
        right_tree, key_right = BuildTree(L, new_root, key_left)
    else:
        right_tree, key_right = left_tree, key_left

    # --- 3. Merge bounds ---
    theta_minus = left_tree.theta_minus if root.v == -1 else right_tree.theta_minus
    r_minus     = left_tree.r_minus     if root.v == -1 else right_tree.r_minus
    theta_plus  = right_tree.theta_plus if root.v == 1  else left_tree.theta_plus
    r_plus      = right_tree.r_plus     if root.v == 1  else left_tree.r_plus

    # --- 4. Choose theta_prime ---
    key_right, sub = jrnd.split(key_right)
    total_n = left_tree.n_prime + right_tree.n_prime
    p = jnp.where(total_n > 0, right_tree.n_prime / total_n, 0.5)
    choose = jrnd.bernoulli(sub, p)
    theta_prime = jnp.where(choose, right_tree.theta_prime, left_tree.theta_prime)

    # --- 5. Stats ---
    n_prime = left_tree.n_prime + right_tree.n_prime
    s_prime = left_tree.s_prime * right_tree.s_prime * stop_criterion(
        theta_minus, theta_plus, r_minus, r_plus
    )
    alpha_prime = left_tree.alpha_prime + right_tree.alpha_prime
    n_a_prime = left_tree.n_a_prime + right_tree.n_a_prime

    merged = Tree(
        theta_minus=theta_minus,
        r_minus=r_minus,
        theta_plus=theta_plus,
        r_plus=r_plus,
        theta_prime=theta_prime,
        n_prime=n_prime,
        s_prime=s_prime,
        alpha_prime=alpha_prime,
        n_a_prime=n_a_prime,
    )

    return merged, key_right



In [6]:
'''
# algorithm 6 No-U-Turn Sampler with Dual Averaging 
def NUTS(
    L         : Callable,
    theta_0   : jnp.ndarray,
    delta     : float,
    max_iter  : int,
    M_adapt   : int,
    key       : jnp.ndarray
    ) -> Tree :

    # set epsilon
    epsilon = FindReasonableEpsilon( L, theta_0, key )

    # setting starting default values [...]
    mu        = jnp.log( 10 * epsilon )
    epsilon_m = 1.0
    H_m       = 0.0
    gamma     = 0.05
    t_0       = 10
    kappa     = 0.75

    # ------------------------
    # for now in plane python
    # ------------------------
    
    for m in range( 1, max_iter ) :
        # sample momentum r ~ N(0, I)
        # split keys
        key, subkey = jrnd.split(key)
        r_0 = jrnd.normal(subkey, shape = theta_0.shape )
        key, subkey = jrnd.split(key)
        u   = jrnd.uniform(subkey, maxval = log_p( L, theta_0, r_0) )
        
        # initialize variables 
        theta_minus = theta_0
        theta_plus  = theta_0
        r_minus     = r_0
        r_plus      = r_0
        j           = jnp.array(0, dtype=jnp.int32)
        theta_prime = theta_0
        n           = jnp.array(1, dtype=jnp.int32)
        s           = jnp.array(1, dtype=jnp.int32)

        while s == 1 :
            # choose a direction v_j ~ uniform[-1;1]
            key, subkey = jrnd.split(key)
            v_j = jnp.where(jrnd.uniform(subkey) < 0.5, -1, 1)

            if v_j == -1:
                root = Root(theta_minus, r_minus, u, v_j, j, epsilon, theta_0, r_0)
            else:
                root = Root(theta_plus,  r_plus,  u, v_j, j, epsilon, theta_0, r_0)

            # Call the BuildTree function
            tree, key = BuildTree(root, key)

            if tree.s_prime == 1 :
                choose = jnp.min( 1, tree.n_prime / n )
                theta_prime = jnp.where(choose, tree.theta_prime, theta_prime)
        
            if v_j == -1:
                theta_minus, r_minus = tree.theta_minus, tree.r_minus
            else:
                theta_plus,  r_plus  = tree.theta_plus,  tree.r_plus
            n += tree.n_prime
            s = tree.s_prime * stop_criterion(
                theta_minus, theta_plus, r_minus, r_plus )
            j += 1
        if m <= M_adapt :
            H_m       = ( 1 - (1/(m + t_0)) ) * H_m + ((delta - tree.alpha_prime/tree.n_a_prime)/(m + t_0))
            epsilon   = jnp.exp( mu - jnp.sqrt(m) / gamma * H_m )
            epsilon_m = jnp.exp( jnp.pow( m, -kappa )*jnp.log(epsilon) + (1-jnp.pow(m, -kappa)*jnp.log(epsilon_m)))
        else :
            epsilon   = epsilon_m
'''

'\n# algorithm 6 No-U-Turn Sampler with Dual Averaging \ndef NUTS(\n    L         : Callable,\n    theta_0   : jnp.ndarray,\n    delta     : float,\n    max_iter  : int,\n    M_adapt   : int,\n    key       : jnp.ndarray\n    ) -> Tree :\n\n    # set epsilon\n    epsilon = FindReasonableEpsilon( L, theta_0, key )\n\n    # setting starting default values [...]\n    mu        = jnp.log( 10 * epsilon )\n    epsilon_m = 1.0\n    H_m       = 0.0\n    gamma     = 0.05\n    t_0       = 10\n    kappa     = 0.75\n\n    # ------------------------\n    # for now in plane python\n    # ------------------------\n\n    for m in range( 1, max_iter ) :\n        # sample momentum r ~ N(0, I)\n        # split keys\n        key, subkey = jrnd.split(key)\n        r_0 = jrnd.normal(subkey, shape = theta_0.shape )\n        key, subkey = jrnd.split(key)\n        u   = jrnd.uniform(subkey, maxval = log_p( L, theta_0, r_0) )\n\n        # initialize variables \n        theta_minus = theta_0\n        theta_plus 